In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/EVSE.zip" -d /content/drive/MyDrive/

In [ ]:
import pandas as pd

df_net = pd.read_csv(
    "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-A/csv/EVSE-A-charging-Aggressive-scan.csv"
)

print(df_net.shape)
print(df_net.columns)
df_net.head()

In [ ]:
df_host = pd.read_csv(
    "/content/drive/MyDrive/EVSE/Host Events/EVSE-B-HPC-Kernel-Events-Combined.csv"
)

print(df_host.shape)
print(df_host.columns)
df_host.head()

In [ ]:
df_power = pd.read_csv(
    "/content/drive/MyDrive/EVSE/Power Consumption/EVSE-B-PowerCombined.csv"
)

print(df_power.shape)
print(df_power.columns)
df_power.head()

In [ ]:
# =========================
# EVSE-A CLEANING PIPELINE
# =========================

import os
import pandas as pd
import numpy as np

# ---- PATHS ----
EVSE_A_PATH = "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-A/csv" # Corrected path
OUTPUT_PATH = "/content/drive/MyDrive/EVSE/cleaned"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---- Helper function to extract label from filename ----
def extract_label_from_filename(filename):
    name_without_ext = os.path.splitext(filename)[0] # remove .csv
    parts = name_without_ext.split('-')

    # Assuming the label is the part(s) after 'charging' or 'idle'
    if 'charging' in parts:
        start_index = parts.index('charging') + 1
    elif 'idle' in parts:
        start_index = parts.index('idle') + 1
    else:
        return 'UNKNOWN' # Fallback for unexpected filenames

    label_parts = parts[start_index:]
    return '-'.join(label_parts).replace(' ', '_').upper()

# ---- LOAD & CONCATENATE ALL EVSE-A CSVs ----
dfs = []
for f in sorted(os.listdir(EVSE_A_PATH)):
    if f.endswith(".csv"):
        file_path = os.path.join(EVSE_A_PATH, f)
        temp_df = pd.read_csv(file_path)
        temp_df['Attack_Type'] = extract_label_from_filename(f) # Add new label column
        dfs.append(temp_df)

df_a = pd.concat(dfs, ignore_index=True)
print("EVSE-A raw shape:", df_a.shape)

# ---- IDENTIFY LABEL COLUMN (AUTO) ----
# Now, 'Attack_Type' should be found
label_candidates = [c for c in df_a.columns if any(k in c.lower() for k in ["label", "attack", "class", "attack_type"])]
assert len(label_candidates) == 1, f"Expected 1 label column, found: {label_candidates}"
LABEL_COL = label_candidates[0]
print("Label column:", LABEL_COL)

# ---- DROP NON-FEATURE (IDENTIFIER) COLUMNS ----
non_feature_cols = [
    'id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui',
    'dst_ip', 'dst_mac', 'dst_oui',
    'application_name', 'application_category_name',
    'requested_server_name', 'client_fingerprint', 'server_fingerprint',
    'user_agent', 'content_type'
]
df_a.drop(columns=non_feature_cols, inplace=True, errors="ignore")
print("Dropped non-feature columns:", non_feature_cols)

# ---- HANDLE INF / NAN ----
df_a.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_cols = df_a.select_dtypes(include="number").columns
# Calculate medians; if a column is all NaN, its median will be NaN
medians = df_a[numeric_cols].median()
# Fill NaNs using medians; if a median is NaN, use 0 as fallback
df_a[numeric_cols] = df_a[numeric_cols].fillna(medians.fillna(0))

# ---- ENSURE LABEL IS CLEAN ----
df_a[LABEL_COL] = df_a[LABEL_COL].astype(str).str.strip().str.upper()

# (Optional) Binary labels example:
# df_a[LABEL_COL] = df_a[LABEL_COL].apply(lambda x: 0 if x == "BENIGN" else 1)

# ---- FINAL SANITY CHECKS ----
assert not df_a[numeric_cols].isna().any().any(), "NaNs remain in numeric features after imputation."
assert not np.isinf(df_a[numeric_cols]).any().any(), "Infs remain in numeric features"

print("EVSE-A cleaned shape:", df_a.shape)
print(df_a[LABEL_COL].value_counts().head())

# ---- SAVE CLEAN DATASET ----
out_a = os.path.join(OUTPUT_PATH, "EVSE-A_clean.csv")
df_a.to_csv(out_a, index=False)
print("Saved:", out_a)


EVSE-A raw shape: (547854, 87)
Label column: Attack_Type
Dropped non-feature columns: ['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'dst_ip', 'dst_mac', 'dst_oui', 'application_name', 'application_category_name', 'requested_server_name', 'client_fingerprint', 'server_fingerprint', 'user_agent', 'content_type']
EVSE-A cleaned shape: (547854, 72)
Attack_Type
SYN-FLOOD         131107
SYNONYMOUS-IP     131093
TCP-FLOOD         131087
PUSH-ACK-FLOOD     65536
UDP-FLOOD          15956
Name: count, dtype: int64
Saved: /content/drive/MyDrive/EVSE/cleaned/EVSE-A_clean.csv


In [ ]:
# =========================
# EVSE-B CLEANING PIPELINE
# =========================

import os
import pandas as pd
import numpy as np

# ---- PATHS ----
EVSE_B_PATH = "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-B/csv" # Corrected path
OUTPUT_PATH = "/content/drive/MyDrive/EVSE/cleaned"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---- Helper function to extract label from filename ----
def extract_label_from_filename(filename):
    name_without_ext = os.path.splitext(filename)[0] # remove .csv
    parts = name_without_ext.split('-')

    # Assuming the label is the part(s) after 'charging' or 'idle'
    if 'charging' in parts:
        start_index = parts.index('charging') + 1
    elif 'idle' in parts:
        start_index = parts.index('idle') + 1
    else:
        return 'UNKNOWN' # Fallback for unexpected filenames

    label_parts = parts[start_index:]
    return '-'.join(label_parts).replace(' ', '_').upper()

# ---- LOAD & CONCATENATE ALL EVSE-B CSVs ----
dfs = []
for f in sorted(os.listdir(EVSE_B_PATH)):
    if f.endswith(".csv"):
        file_path = os.path.join(EVSE_B_PATH, f)
        temp_df = pd.read_csv(file_path)
        temp_df['Attack_Type'] = extract_label_from_filename(f) # Add new label column
        dfs.append(temp_df)

df_b = pd.concat(dfs, ignore_index=True)
print("EVSE-B raw shape:", df_b.shape)

# ---- IDENTIFY LABEL COLUMN (AUTO) ----
# Now, 'Attack_Type' should be found
label_candidates = [c for c in df_b.columns if any(k in c.lower() for k in ["label", "attack", "class", "attack_type"])]
assert len(label_candidates) == 1, f"Expected 1 label column, found: {label_candidates}"
LABEL_COL = label_candidates[0]
print("Label column:", LABEL_COL)

# ---- DROP NON-FEATURE (IDENTIFIER) COLUMNS ----
non_feature_cols = [
    'id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui',
    'dst_ip', 'dst_mac', 'dst_oui',
    'application_name', 'application_category_name',
    'requested_server_name', 'client_fingerprint', 'server_fingerprint',
    'user_agent', 'content_type'
]
df_b.drop(columns=non_feature_cols, inplace=True, errors="ignore")
print("Dropped non-feature columns:", non_feature_cols)

# ---- HANDLE INF / NAN ----
df_b.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_cols = df_b.select_dtypes(include="number").columns
# Calculate medians; if a column is all NaN, its median will be NaN
medians = df_b[numeric_cols].median()
# Fill NaNs using medians; if a median is NaN, use 0 as fallback
df_b[numeric_cols] = df_b[numeric_cols].fillna(medians.fillna(0))

# ---- ENSURE LABEL IS CLEAN ----
df_b[LABEL_COL] = df_b[LABEL_COL].astype(str).str.strip().str.upper()

# (Optional) Binary labels example:
# df_b[LABEL_COL] = df_b[LABEL_COL].apply(lambda x: 0 if x == "BENIGN" else 1)

# ---- FINAL SANITY CHECKS ----
assert not df_b[numeric_cols].isna().any().any(), "NaNs remain in numeric features after imputation."
assert not np.isinf(df_b[numeric_cols]).any().any(), "Infs remain in numeric features"

print("EVSE-B cleaned shape:", df_b.shape)
print(df_b[LABEL_COL].value_counts().head())

# ---- SAVE CLEAN DATASET ----
out_b = os.path.join(OUTPUT_PATH, "EVSE-B_clean.csv")
df_b.to_csv(out_b, index=False)
print("Saved:", out_b)

EVSE-B raw shape: (2196846, 87)
Label column: Attack_Type
Dropped non-feature columns: ['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'dst_ip', 'dst_mac', 'dst_oui', 'application_name', 'application_category_name', 'requested_server_name', 'client_fingerprint', 'server_fingerprint', 'user_agent', 'content_type']
EVSE-B cleaned shape: (2196846, 72)
Attack_Type
UNKNOWN                   564781
PORT-SCAN                 255554
SYN-STEALTH               214065
SERVICE-DETECTION-SCAN    165171
VULNERABILITY-SCAN        161728
Name: count, dtype: int64
Saved: /content/drive/MyDrive/EVSE/cleaned/EVSE-B_clean.csv


In [ ]:
import pandas as pd
import numpy as np

A_PATH = "/content/drive/MyDrive/EVSE/cleaned/EVSE-A_clean.csv"
B_PATH = "/content/drive/MyDrive/EVSE/cleaned/EVSE-B_clean.csv"

df_a = pd.read_csv(A_PATH)
df_b = pd.read_csv(B_PATH)

print("EVSE-A shape:", df_a.shape)
print("EVSE-B shape:", df_b.shape)

# Identify label column automatically
label_candidates = [c for c in df_a.columns if any(k in c.lower() for k in ["label","attack","class"])]
assert len(label_candidates)==1, label_candidates
LABEL = label_candidates[0]
print("Label column:", LABEL)

df_a[LABEL].value_counts().head(), df_b[LABEL].value_counts().head()

EVSE-A shape: (547854, 72)
EVSE-B shape: (2196846, 72)
Label column: Attack_Type


In [ ]:
# Binary labels: BENIGN=0, ATTACK=1
def binarize(y):
    return (y.astype(str).str.upper() != "BENIGN").astype(int)

X_train = df_a.drop(columns=[LABEL])
y_train = binarize(df_a[LABEL])

X_test  = df_b.drop(columns=[LABEL])
y_test  = binarize(df_b[LABEL])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train attack rate:", y_train.mean(), "Test attack rate:", y_test.mean())

In [ ]:
print("df_a Attack_Type value counts:")
print(df_a['Attack_Type'].value_counts())
print("\n" + "df_b Attack_Type value counts:")
print(df_b['Attack_Type'].value_counts())

df_a Attack_Type value counts:
Attack_Type
SYN-FLOOD             131107
SYNONYMOUS-IP         131093
TCP-FLOOD             131087
PUSH-ACK-FLOOD         65536
UDP-FLOOD              15956
PORTSCAN               13092
VULNERABILITY-SCAN     12581
AGGRESSIVE-SCAN        12332
OS-FINGERPRINTING       8381
SYN-STEALTH             8032
SYN-STEALTH-SCAN        7215
SERVICE-DETECTION       7141
SLOWLORIS-SCAN          4201
BENIGN                    68
ICMP-FRAGMENTATION        16
UNKNOWN                   14
ICMP-FLOOD                 2
Name: count, dtype: int64

df_b Attack_Type value counts:
Attack_Type
UNKNOWN                   564781
PORT-SCAN                 255554
SYN-STEALTH               214065
SERVICE-DETECTION-SCAN    165171
VULNERABILITY-SCAN        161728
PUSH-ACK-FLOOD            131118
TCP-FLOOD                 131107
SYNONYMOUS-IP-FLOOD       131092
SYN-FLOOD                 131089
OS-FINGERPRINTING         113633
AGGRESSIVE-SCAN            73829
SYN-STEALTH-SCAN           7363

In [ ]:
import pandas as pd
import numpy as np

A_PATH = "/content/drive/MyDrive/EVSE/cleaned/EVSE-A_clean.csv"
B_PATH = "/content/drive/MyDrive/EVSE/cleaned/EVSE-B_clean.csv"

df_a = pd.read_csv(A_PATH)
df_b = pd.read_csv(B_PATH)

# Detect label column automatically
label_candidates = [c for c in df_a.columns if any(k in c.lower() for k in ["label","attack","class"])]
assert len(label_candidates) == 1, label_candidates
LABEL = label_candidates[0]

print("Label column:", LABEL)
print("EVSE-A shape:", df_a.shape)
print("EVSE-B shape:", df_b.shape)


Label column: Attack_Type
EVSE-A shape: (547854, 72)
EVSE-B shape: (2196846, 72)


In [ ]:
# Binary encoding: BENIGN = 0, ATTACK = 1
def binarize(y):
    return (y.astype(str).str.upper() != "BENIGN").astype(int)

X_train = df_a.drop(columns=[LABEL])
y_train = binarize(df_a[LABEL])

X_test  = df_b.drop(columns=[LABEL])
y_test  = binarize(df_b[LABEL])

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train attack ratio:", y_train.mean())
print("Test attack ratio:", y_test.mean())


Train shape: (547854, 71) Test shape: (2196846, 71)
Train attack ratio: 0.9998758793401161
Test attack ratio: 1.0
